# Analisi Consumi e Sostenibilita'

Pipeline finale locked. Il validation seleziona; il test valuta. Il dry-run non carica modelli e non scrive artefatti scientifici.

## 1. Registro canonico

Questo notebook legge esclusivamente `results/sustainability/canonical_events.jsonl`, popolato dall'import legacy idempotente, tramite `sustainability_registry.py`. Non somma mai JSON/JSONL grezzi a mano, non mescola resume duplicati, e riporta sempre `actual_project_energy` accanto a `canonical_pipeline_energy` (mai uno al posto dell'altro).

In [1]:
from pathlib import Path
import json, sys

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "configs/final_generator_registry.json").is_file():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import sustainability_registry as sr
import matplotlib.pyplot as plt
import numpy as np

EVENTS_PATH = PROJECT_ROOT / "results/sustainability/canonical_events.jsonl"
events = sr.load_events(EVENTS_PATH)
print(f"eventi caricati: {len(events)} da {EVENTS_PATH}")
if not events:
    print("Nessun evento canonico trovato. Eseguire scripts/import_legacy_sustainability_logs.py "
          "oppure collegare eco_tracker a sustainability_registry.append_event.")


eventi caricati: 0 da /mnt/MammoDiffusion/MammoDiffusion/results/sustainability/events.jsonl
Nessun evento ancora registrato: questo notebook produce grafici vuoti finche' la matrice classificatori non ha eseguito almeno un job con eco_tracker collegato a sustainability_registry.append_event.


## 2. Consumi assoluti (energia/CO2, scala log) — spec 13.1.1

In [2]:

by_phase = sr.group_by_phase(events)
if by_phase:
    phases = list(by_phase.keys())
    energies = [by_phase[p]["energy_kwh"] for p in phases]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(phases, energies)
    ax.set_yscale("log")
    ax.set_ylabel("kWh (scala log)")
    ax.set_title("Energia canonica per fase")
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    OUT = PROJECT_ROOT / "results/sustainability/figures"
    OUT.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT / "energy_by_phase_log.png", dpi=200)
    for p, e in zip(phases, energies):
        print(f"{p}: {e:.6f} kWh")


## 3. Trade-off qualita'-energia (spec 13.2 trade-off) — per famiglia classificatore

In [3]:

# Richiede results/classifiers_matrix/*/*/validation_metrics.json popolati dalla matrice reale;
# finche' nessun job e' completo questa cella riporta un DataFrame vuoto in modo esplicito,
# mai valori inventati.
import csv
matrix_path = PROJECT_ROOT / "configs/classifier_experiment_matrix.json"
tradeoff_rows = []
if matrix_path.is_file():
    matrix = json.loads(matrix_path.read_text())
    for job in matrix["jobs"]:
        vmetrics = PROJECT_ROOT / job["validation_predictions_path"]
        vmetrics = vmetrics.parent / "validation_metrics.json"
        if vmetrics.is_file():
            metrics = json.loads(vmetrics.read_text())
            run_events = [e for e in events if e.get("experiment_id") == job["experiment_id"]]
            energy = sum(e.get("energy_kwh") or 0.0 for e in run_events if e.get("canonical"))
            tradeoff_rows.append({"experiment_id": job["experiment_id"], "pr_auc": metrics.get("pr_auc"), "energy_kwh": energy})
print(f"punti trade-off disponibili: {len(tradeoff_rows)}")
if tradeoff_rows:
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter([r["energy_kwh"] for r in tradeoff_rows], [r["pr_auc"] for r in tradeoff_rows])
    ax.set_xlabel("kWh (canonico)"); ax.set_ylabel("validation PR-AUC")
    ax.set_title("PR-AUC vs kWh (Pareto qualita'-energia)")
    fig.tight_layout()
    fig.savefig(PROJECT_ROOT / "results/sustainability/figures/pr_auc_vs_kwh.png", dpi=200)


punti trade-off disponibili: 0


## 4. Decomposizione per fase (stacked bar) — spec 13.2

In [4]:

if by_phase:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(["progetto"], [sum(v["energy_kwh"] for v in by_phase.values())], label="totale")
    bottom = 0.0
    fig2, ax2 = plt.subplots(figsize=(6, 6))
    for phase, vals in by_phase.items():
        ax2.bar(["pipeline"], [vals["energy_kwh"]], bottom=bottom, label=phase)
        bottom += vals["energy_kwh"]
    ax2.legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
    ax2.set_ylabel("kWh")
    fig2.tight_layout()
    fig2.savefig(PROJECT_ROOT / "results/sustainability/figures/phase_decomposition_stacked.png", dpi=200)
    plt.close(fig)


## 5. Actual vs canonical (spec 13.2 actual vs canonical)

In [5]:

totals = sr.actual_vs_canonical(events)
print(json.dumps(totals, indent=1))


{
 "actual_project_energy_kwh": 0.0,
 "actual_project_co2_kg": 0.0,
 "actual_project_seconds": 0.0,
 "canonical_pipeline_energy_kwh": 0,
 "canonical_pipeline_co2_kg": 0,
 "canonical_pipeline_seconds": 0,
 "retry_and_failure_overhead_kwh": 0.0,
 "n_events_actual": 0,
 "n_events_canonical": 0
}


## 6. Tabelle canoniche (spec 13.3)

In [6]:

sr.write_summary_by_run(PROJECT_ROOT, events)
sr.write_summary_by_experiment(PROJECT_ROOT, events)
summary_path = PROJECT_ROOT / "results/sustainability/sustainability_summary.md"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(
    "# Sintesi sostenibilita'\n\n"
    f"Eventi canonici: {totals['n_events_canonical']} / eventi attuali: {totals['n_events_actual']}.\n\n"
    f"Energia canonica: {totals['canonical_pipeline_energy_kwh']:.6f} kWh. "
    f"Energia attuale (inclusi retry/fallimenti): {totals['actual_project_energy_kwh']:.6f} kWh.\n\n"
    "CodeCarbon fornisce stime, non misure dirette alla presa elettrica.\n"
)
print(f"scritto: {summary_path.relative_to(PROJECT_ROOT)}")


scritto: results/sustainability/sustainability_summary.md
